In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import torch.optim as optim
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from torch.utils.data import random_split
import numpy as np
import matplotlib as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


Bu model zik zak çiziyor. Batch size, lr kontrol edilecek. Gerekirse de early stopping eklenecek

In [2]:
BLUE = "\033[94m"
GREEN = "\033[92m"
MAGENTA = "\033[95m"
CYAN = "\033[96m"
YELLOW = "\033[93m"
RESET = "\033[0m"

In [3]:
batch_size = 64
lr = 0.001
patience = 5 #early stopping eklersem

In [4]:
cifar_transform = transforms.Compose([transforms.Grayscale(num_output_channels=1),
                                      transforms.Resize((28, 28)),
                                      transforms.ToTensor(),
                                      transforms.Normalize((0.5,), (0.5,))])

transfrom = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize((0.5,), (0.5,))])

cifar10 = datasets.CIFAR10(root='./data', train=True, download=True, transform=cifar_transform)
mnist = datasets.MNIST(root='./data', train=True, download=True, transform=transfrom)


In [5]:
class MNISTDataset(Dataset):
    def __init__(self, mnist_data):
        self.mnist_data = mnist_data

    def __len__(self):
        return len(self.mnist_data)

    def __getitem__(self, idx):
        img, label = self.mnist_data[idx]
        return img, label # Flatten the image to a 1D tensor

In [6]:
class CIFAR10Dataset(Dataset):
    def __init__(self, cifar_data):
        self.cifar_data = cifar_data

    def __len__(self):
        return len(self.cifar_data)

    def __getitem__(self, idx):
        img, label = self.cifar_data[idx]
        return img, label #olunca 784 olmayınca 1, 28, 28 !!

In [7]:
cifar10_dataset = CIFAR10Dataset(cifar10)
mnist_dataset = MNISTDataset(mnist)

In [8]:
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

cifar10_len = len(cifar10_dataset)
mnist_len = len(mnist_dataset)
train_len_cifar = int(train_ratio * cifar10_len)
val_len_cifar = int(val_ratio * cifar10_len)

train_len_mnist = int(train_ratio * mnist_len)
val_len_mnist = int(val_ratio * mnist_len)

test_len_cifar = cifar10_len - train_len_cifar - val_len_cifar
test_len_mnist = mnist_len - train_len_mnist - val_len_mnist

dataset_train_cifar, dataset_val_cifar, dataset_test_cifar = random_split(cifar10_dataset, [train_len_cifar, val_len_cifar, test_len_cifar])
dataset_train_mnist, dataset_val_mnist, dataset_test_mnist = random_split(mnist_dataset, [train_len_mnist, val_len_mnist, test_len_mnist])


In [9]:

train_loader_cifar = DataLoader(dataset_train_cifar, batch_size=batch_size, shuffle=True)
val_loader_cifar = DataLoader(dataset_val_cifar, batch_size=batch_size, shuffle=False)
test_loader_cifar = DataLoader(dataset_test_cifar, batch_size=1, shuffle=False)


train_loader_mnist = DataLoader(dataset_train_mnist, batch_size=batch_size, shuffle=True)
val_loader_mnist = DataLoader(dataset_val_mnist, batch_size=batch_size, shuffle=False)
test_loader_mnist = DataLoader(dataset_test_mnist, batch_size=1, shuffle=False)

In [10]:
class StepFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input, treshold=0.0):
        ctx.save_for_backward(input)
        return (input > treshold).float()

    @staticmethod
    def backward(ctx, grad_output):
        input, = ctx.saved_tensors
        grad_input = grad_output.clone()
        #grad_input[input <= 0] = 0
        return grad_input, None
    
def step_function(input, treshold=0.0):
    return StepFunction.apply(input, treshold)

def soft_gate(logits, epoch, warmup=3):
    if epoch < warmup:
        sharpness = (epoch + 1) / warmup * 5 
        #return torch.sigmoid(logits * (epoch/warmup))
        return torch.sigmoid(logits * sharpness)
    return step_function(logits)

In [11]:
def train_model(model, train_loader, val_loader, epochs):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        train_acc = 0.0
        
        correct = 0
        total = 0

        for inputs, labels in tqdm(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        model.eval()
        valid_loss = 0.0
        valid_acc = 0.0
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for data, target in tqdm(val_loader):
                data, target = data.to(device), target.to(device)
                output = model(data)# aslında model.forward(data), tahmin üretir

                loss_val = criterion(output, target)#buna neden gerek duyduk ki?
                valid_loss += loss_val.item() * data.size(0)
                _, predicted = torch.max(output.data, 1)
                val_correct += (predicted == target).sum().item()
                val_total += labels.size(0)


        train_loss = running_loss / total
        valid_loss = valid_loss / val_total

        train_acc = correct / total
        valid_acc = val_correct / val_total


        print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {train_loss:.4f}, Val Loss: {valid_loss:.4f}, Train Acc: {train_acc:.4f}, Val Acc: {valid_acc:.4f}')


In [12]:
class Model1(nn.Module):
    def __init__(self, input_channels, num_classes):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.3)  # Dropout layer for regularization
        )

        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.encoder(x)
        x = self.classifier(x)
        return x

In [13]:
model1_mnist = Model1(input_channels=1, num_classes=10).to(device)
model1_cifar = Model1(input_channels=1, num_classes=10).to(device)

In [14]:
print(f"{CYAN}CIFAR10 pretraining started...{RESET}")
pretrained_cifar = train_model(model1_cifar, train_loader_cifar, val_loader_cifar, epochs=10)
print(f"{CYAN}CIFAR10 pretraining completed.{RESET}")
print(f"{MAGENTA}MNIST pretraining started...{RESET}")
pretrained_mnist = train_model(model1_mnist, train_loader_mnist, val_loader_mnist, epochs=10)
print(f"{MAGENTA}MNIST pretraining completed.{RESET}")

CIFAR10 pretraining started...


100%|██████████| 118/118 [00:01<00:00, 105.45it/s]


Epoch [1/10], Train Loss: 1.7427, Val Loss: 1.6785, Train Acc: 0.3686, Val Acc: 0.5375


100%|██████████| 118/118 [00:01<00:00, 106.78it/s]


Epoch [2/10], Train Loss: 1.4296, Val Loss: 1.4749, Train Acc: 0.4914, Val Acc: 0.6137


...

Epoch [8/10], Train Loss: 0.0216, Val Loss: 0.1377, Train Acc: 0.9930, Val Acc: 3.9552


100%|██████████| 141/141 [00:01<00:00, 133.43it/s]


Epoch [9/10], Train Loss: 0.0191, Val Loss: 0.1834, Train Acc: 0.9934, Val Acc: 3.9490


100%|██████████| 141/141 [00:01<00:00, 130.71it/s]

Epoch [10/10], Train Loss: 0.0182, Val Loss: 0.1503, Train Acc: 0.9940, Val Acc: 3.9526
MNIST pretraining completed.


New Dataloaders for Combined Datasets

In [15]:
combined_dataset = ConcatDataset([cifar10_dataset, mnist_dataset])


train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

total_len = len(combined_dataset)
train_len = int(train_ratio * total_len)
val_len = int(val_ratio * total_len)
test_len = total_len - train_len - val_len

dataset_train, dataset_val, dataset_test = random_split(combined_dataset, [train_len, val_len, test_len])

In [20]:
#silinecek yüksek ihtimalle
class AutoGatedModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.mnist_encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64*7*7, 128)
        )
            
        self.cifar_encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64*7*7, 128)  # Adjusted for CIFAR-10 input size
        )
        
        # Gate network
        self.gate = nn.Sequential(
            nn.Linear(128 * 2, 256),
            nn.LeakyReLU(0.2),
            nn.LayerNorm(256),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
            nn.LayerNorm(128),
            nn.Linear(128, 1)
        )
        # Classifier
        self.classifier = nn.Linear(128, 10)

    def forward(self, x1, x2):
        # Extract features from both encoders
        mnist_feat = self.mnist_encoder(x1)
        cifar_feat = self.cifar_encoder(x2)
        
        # Gate decision
        combined = torch.cat([mnist_feat, cifar_feat], dim=1)
        gate_logits = self.gate(combined)
        gate = torch.sigmoid(gate_logits)
        
        # Gated combination
        features = gate * mnist_feat + (1-gate) * cifar_feat
        
        # Classification
        logits = self.classifier(features)
        
        return {
            'logits': logits,
            'gate': gate,
            'features': features
        }

In [ ]:
#silinecek yüksek ihtimalle
def train_autogate(model, train_loader, val_loader, epochs=10):
    opt = optim.AdamW(model.parameters(), lr=lr)#lr=3e-4
    criterion = nn.CrossEntropyLoss()
    
    best_val_acc = 0
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0
        train_gates = []
        
        for x1, x2, y in tqdm(train_loader, desc=f"Train Epoch {epoch+1}"):
            x1, x2 = x1.to(device), x2.to(device)
            y = y.to(device)
            
            opt.zero_grad()
            outputs = model(x1, x2)
            
            # Classification loss
            loss = criterion(outputs['logits'], y)
            
            # Gate balance regularization
            gate_mean = outputs['gate'].mean()
            balance_loss = torch.abs(gate_mean - 0.5)
            
            # Feature diversity
            '''mnist_feat = model.mnist_encoder(outputs['features'])
            cifar_feat = model.cifar_encoder(outputs['features'])
            div_loss = -F.cosine_similarity(mnist_feat, cifar_feat).mean()'''
            
            gate_probs = torch.sigmoid(gate_mean)
            entropy_loss = - (gate_probs * torch.log(gate_probs + 1e-10)).mean()
            # Total loss
            total_loss = loss + 0.5*balance_loss + 0.1*entropy_loss
            total_loss.backward()
            opt.step()
            
            # Metrics
            train_loss += loss.item()
            preds = outputs['logits'].argmax(1)
            train_correct += (preds == y).sum().item()
            train_total += y.size(0)
            train_gates.extend(outputs['gate'].squeeze().tolist())
        
        # Validation phase
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        val_gates = []
        
        with torch.no_grad():
            for x1, x2, y in tqdm(val_loader, desc="Validating"):
                x1, x2 = x1.to(device), x2.to(device)
                y = y.to(device)
                outputs = model(x1, x2)
                
                val_loss += criterion(outputs['logits'], y).item()
                preds = outputs['logits'].argmax(1)
                val_correct += (preds == y).sum().item()
                val_total += y.size(0)
                val_gates.extend(outputs['gate'].squeeze().tolist())
        
        # Calculate metrics
        train_acc = train_correct / train_total
        val_acc = val_correct / val_total
        #train ve val için ayrı ayrı gate'e ne gerek var?
        avg_train_gate = np.mean(train_gates)
        avg_val_gate = np.mean(val_gates)

        #buraya if else eklenecek: >0.5 ise mnsit, <0.5 ise cifar, =0.5 ise model kararsız şeklinde
        #sorun var, asla tam olarak 0.5 gelmeyecek
        #train ve val gate olarak ayırmak anlamsız geldi bunu nasıl birleştirebilirim bak!
        if avg_train_gate > 0.5:
            print(f"\nEpoch {epoch+1}/{epochs}:")
            print(f"  Train Loss: {train_loss/len(train_loader):.4f} | Acc: {train_acc:.4f} | Gate: {avg_train_gate:.3f}")
            print(f"{GREEN}Training gate is biased towards MNIST.{RESET}")
        elif avg_train_gate < 0.5:
            print(f"\nEpoch {epoch+1}/{epochs}:")
            print(f"  Train Loss: {train_loss/len(train_loader):.4f} | Acc: {train_acc:.4f} | Gate: {avg_train_gate:.3f}")
            print(f"{CYAN}Training gate is biased towards CIFAR-10.{RESET}")
        else:
            print(f"\nEpoch {epoch+1}/{epochs}:")
            print(f"  Train Loss: {train_loss/len(train_loader):.4f} | Acc: {train_acc:.4f} | Gate: {avg_train_gate:.3f}")
            print(f"{YELLOW}Training gate is uncertain.{RESET}")

        if avg_val_gate > 0.5:
            print(f"\nEpoch {epoch+1}/{epochs}:")
            print(f"  Val Loss: {val_loss/len(val_loader):.4f} | Acc: {val_acc:.4f} | Gate: {avg_val_gate:.3f}")
            print(f"{GREEN}Validation gate is biased towards MNIST.{RESET}")
        elif avg_val_gate < 0.5:
            print(f"\nEpoch {epoch+1}/{epochs}:")
            print(f"  Val Loss: {val_loss/len(val_loader):.4f} | Acc: {val_acc:.4f} | Gate: {avg_val_gate:.3f}")
            print(f"{CYAN}Validation gate is biased towards CIFAR-10.{RESET}")
        else:
            print(f"\nEpoch {epoch+1}/{epochs}:")
            print(f"  Val Loss: {val_loss/len(val_loader):.4f} | Acc: {val_acc:.4f} | Gate: {avg_val_gate:.3f}")
            print(f"{YELLOW}Validation gate is uncertain.{RESET}")
        
        # Print epoch summary
        '''print(f"\nEpoch {epoch+1}/{epochs}:")
        print(f"  Train Loss: {train_loss/len(train_loader):.4f} | Acc: {train_acc:.4f} | Gate: {avg_train_gate:.3f}")
        print(f"  Val Loss: {val_loss/len(val_loader):.4f} | Acc: {val_acc:.4f} | Gate: {avg_val_gate:.3f}")'''
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'best_autogate.pth')
            print("  ↳ New best model saved!")
        
        # Print gate distribution
        print("\nGate Distribution:")
        print(f"  Train: {np.histogram(train_gates, bins=[0,0.2,0.4,0.6,0.8,1.0])[0]}")
        print(f"  Val:   {np.histogram(val_gates, bins=[0,0.2,0.4,0.6,0.8,1.0])[0]}")
        print("-"*60)


In [22]:
model2 = AutoGatedModel().to(device)

'''model2.encoder1.load_state_dict(model1_mnist.encoder.state_dict())
# Copy CIFAR encoder weights to Model2.encoder2
model2.encoder2.load_state_dict(model1_cifar.encoder.state_dict())

# 3. Optionally freeze encoders(yapmazsak fine-tuning yapar)
for param in model2.encoder1.parameters():
    param.requires_grad = False
for param in model2.encoder2.parameters():
    param.requires_grad = False'''

'model2.encoder1.load_state_dict(model1_mnist.encoder.state_dict())\n# Copy CIFAR encoder weights to Model2.encoder2\nmodel2.encoder2.load_state_dict(model1_cifar.encoder.state_dict())\n\n# 3. Optionally freeze encoders(yapmazsak fine-tuning yapar)\nfor param in model2.encoder1.parameters():\n    param.requires_grad = False\nfor param in model2.encoder2.parameters():\n    param.requires_grad = False'

In [23]:
# Helper: Custom dataset to yield (x1, x2, label)
class PairedDataset(torch.utils.data.Dataset):
	def __init__(self, dataset):
		self.dataset = dataset

	def __len__(self):
		return len(self.dataset)

	def __getitem__(self, idx):
		# For simplicity, use the same image as both x1 and x2
		# You can customize this logic as needed
		img, label = self.dataset[idx]
		return img, img, label

paired_train_loader = DataLoader(PairedDataset(dataset_train), batch_size=batch_size, shuffle=True)
paired_val_loader = DataLoader(PairedDataset(dataset_val), batch_size=batch_size, shuffle=False)
paired_test_loader = DataLoader(PairedDataset(dataset_test), batch_size=1, shuffle=False)

print("Paired training started...")
train_autogate(model2, paired_train_loader, paired_val_loader, epochs=10)
print("Paired training completed.")

#normalde ilk runladığımda iki datasetten de gate için kullanıyordu ama sonra sadece mnsit için kullanmaya başladı(bakılacak!!!!)

Paired training started...


Validating: 100%|██████████| 258/258 [00:02<00:00, 86.63it/s]



Epoch 1/10:
  Train Loss: 0.8279 | Acc: 0.7112 | Gate: 0.499
Training gate is biased towards CIFAR-10.

Epoch 1/10:
  Val Loss: 0.6607 | Acc: 0.7716 | Gate: 0.486
Validation gate is biased towards CIFAR-10.
  ↳ New best model saved!

Gate Distribution:
  Train: [  328  7898 64925  3849     0]
  Val:   [    0   759 15741     0     0]
------------------------------------------------------------


Validating: 100%|██████████| 258/258 [00:03<00:00, 83.79it/s]



Epoch 2/10:
  Train Loss: 0.5789 | Acc: 0.7984 | Gate: 0.500
Training gate is biased towards MNIST.

Epoch 2/10:
  Val Loss: 0.5610 | Acc: 0.8042 | Gate: 0.501
Validation gate is biased towards MNIST.
  ↳ New best model saved!

Gate Distribution:
  Train: [   54  7516 67671  1759     0]
  Val:   [    0   702 15796     2     0]
------------------------------------------------------------


...


Epoch 8/10:
  Train Loss: 0.3150 | Acc: 0.8906 | Gate: 0.500
Training gate is biased towards MNIST.

Epoch 8/10:
  Val Loss: 0.5635 | Acc: 0.8270 | Gate: 0.492
Validation gate is biased towards CIFAR-10.
  ↳ New best model saved!

Gate Distribution:
  Train: [    0  4970 71510   520     0]
  Val:   [    0  2156 14164   180     0]
------------------------------------------------------------


Validating: 100%|██████████| 258/258 [00:03<00:00, 85.05it/s]



Epoch 9/10:
  Train Loss: 0.2897 | Acc: 0.8995 | Gate: 0.500
Training gate is biased towards CIFAR-10.

Epoch 9/10:
  Val Loss: 0.5841 | Acc: 0.8252 | Gate: 0.504
Validation gate is biased towards MNIST.

Gate Distribution:
  Train: [    0  5564 70871   565     0]
  Val:   [    0   803 15594   103     0]
------------------------------------------------------------


Validating: 100%|██████████| 258/258 [00:03<00:00, 82.74it/s]


Epoch 10/10:
  Train Loss: 0.2633 | Acc: 0.9086 | Gate: 0.500
Training gate is biased towards CIFAR-10.

Epoch 10/10:
  Val Loss: 0.6264 | Acc: 0.8190 | Gate: 0.496
Validation gate is biased towards CIFAR-10.

Gate Distribution:
  Train: [    0  5849 70614   537     0]
  Val:   [    0  1218 15207    75     0]
------------------------------------------------------------
Paired training completed.
